# 🧙‍♂️ Compilation MerlinLLM (Alignement Colab)

Ce notebook compile **merlin_llm** pour Windows avec les paramètres optimisés:
- **Contexte**: 8192 tokens (×4)
- **max_tokens**: 256-512
- **temperature**: 0.7
- **top_k**: 50
- **repetition_penalty**: 1.1

## 📦 Fichiers à préparer sur votre PC

Avant de lancer, **créez un ZIP** contenant:
```
merlin_llm_sources.zip
├── src/
│   ├── merlin_llm.cpp
│   └── register_types.cpp
├── CMakeLists.txt
├── godot-cpp/  (dossier complet)
└── llama.cpp/  (dossier complet)
```

Puis **uploadez** ce ZIP dans Colab quand demandé.

In [ ]:
# Cellule 1: Installation des outils de compilation Windows
print("🔧 Installation de MinGW-w64 (cross-compilateur Windows)...\n")

!apt-get update -qq
!apt-get install -y -qq mingw-w64 cmake ninja-build zip unzip

print("\n✅ Outils installés!")
!x86_64-w64-mingw32-g++ --version | head -1

In [ ]:
# Cellule 2: Upload du ZIP avec vos sources
from google.colab import files
import os

print("📤 Uploadez votre fichier 'merlin_llm_sources.zip'\n")
print("Ce ZIP doit contenir:")
print("  - src/ (merlin_llm.cpp, register_types.cpp)")
print("  - CMakeLists.txt")
print("  - godot-cpp/ (complet)")
print("  - llama.cpp/ (complet)")
print("\nCliquez sur 'Choisir les fichiers' ci-dessous:\n")

uploaded = files.upload()

# Décompression
!mkdir -p /content/merlin_llm
!unzip -q merlin_llm_sources.zip -d /content/merlin_llm

print("\n✅ Sources décompressées!")
!ls -la /content/merlin_llm

In [ ]:
# Cellule 3: Vérification des fichiers sources critiques
print("🔍 Vérification de la structure...\n")

import os

required_files = [
    "/content/merlin_llm/src/merlin_llm.cpp",
    "/content/merlin_llm/src/register_types.cpp",
    "/content/merlin_llm/CMakeLists.txt"
]

required_dirs = [
    "/content/merlin_llm/godot-cpp",
    "/content/merlin_llm/llama.cpp"
]

all_ok = True
for f in required_files:
    if os.path.exists(f):
        print(f"✅ {f}")
    else:
        print(f"❌ MANQUANT: {f}")
        all_ok = False

for d in required_dirs:
    if os.path.isdir(d):
        print(f"✅ {d}/")
    else:
        print(f"❌ MANQUANT: {d}/")
        all_ok = False

if all_ok:
    print("\n✅ Tous les fichiers requis sont présents!")
else:
    print("\n❌ Fichiers manquants! Vérifiez votre ZIP.")
    raise Exception("Structure de fichiers incorrecte")

In [ ]:
# Cellule 4: Compilation de godot-cpp pour Windows
print("🔨 Compilation de godot-cpp pour Windows (x86_64)...\n")

os.chdir("/content/merlin_llm/godot-cpp")

# Vérifier si déjà compilé
if os.path.exists("bin/libgodot-cpp.windows.template_release.x86_64.lib"):
    print("✅ godot-cpp déjà compilé!")
else:
    # Créer toolchain file pour MinGW
    with open("/tmp/mingw-toolchain.cmake", "w") as f:
        f.write("""set(CMAKE_SYSTEM_NAME Windows)
set(CMAKE_C_COMPILER x86_64-w64-mingw32-gcc)
set(CMAKE_CXX_COMPILER x86_64-w64-mingw32-g++)
set(CMAKE_RC_COMPILER x86_64-w64-mingw32-windres)
set(CMAKE_FIND_ROOT_PATH /usr/x86_64-w64-mingw32)
set(CMAKE_FIND_ROOT_PATH_MODE_PROGRAM NEVER)
set(CMAKE_FIND_ROOT_PATH_MODE_LIBRARY ONLY)
set(CMAKE_FIND_ROOT_PATH_MODE_INCLUDE ONLY)
""")
    
    !scons platform=windows target=template_release arch=x86_64 use_mingw=yes 2>&1 | tail -20
    
    if os.path.exists("bin/libgodot-cpp.windows.template_release.x86_64.a"):
        print("\n✅ godot-cpp compilé avec succès!")
    else:
        print("\n❌ Erreur lors de la compilation de godot-cpp")
        raise Exception("godot-cpp compilation failed")

In [ ]:
# Cellule 5: Compilation de llama.cpp pour Windows
print("🦙 Compilation de llama.cpp pour Windows (x86_64)...\n")

os.chdir("/content/merlin_llm/llama.cpp")

if os.path.exists("build/bin/llama.lib"):
    print("✅ llama.cpp déjà compilé!")
else:
    !mkdir -p build && cd build && \
    cmake .. \
        -DCMAKE_TOOLCHAIN_FILE=/tmp/mingw-toolchain.cmake \
        -G Ninja \
        -DCMAKE_BUILD_TYPE=Release \
        -DBUILD_SHARED_LIBS=OFF && \
    ninja -j$(nproc) 2>&1 | tail -30
    
    if os.path.exists("build/bin/llama.lib") or os.path.exists("build/libllama.a"):
        print("\n✅ llama.cpp compilé avec succès!")
    else:
        print("\n❌ Erreur lors de la compilation de llama.cpp")
        !ls -la build/

In [ ]:
# Cellule 6: Modification de merlin_llm.cpp avec paramètres Colab
print("⚙️ Application des paramètres optimisés Colab...\n")

import re

cpp_file = "/content/merlin_llm/src/merlin_llm.cpp"

with open(cpp_file, 'r', encoding='utf-8') as f:
    content = f.read()

# Modifications critiques
modifications = [
    (r'n_ctx\s*=\s*2048', 'n_ctx = 8192'),  # Contexte × 4
    (r'default_max_tokens\s*=\s*150', 'default_max_tokens = 256'),
    (r'default_temperature\s*=\s*0\.35', 'default_temperature = 0.7'),
]

# Ajouter top_k et repetition_penalty si absents
if 'top_k' not in content:
    content = content.replace(
        'llama_sampling_params sparams = llama_sampling_default_params();',
        'llama_sampling_params sparams = llama_sampling_default_params();\n\tsparams.top_k = 50;\n\tsparams.penalty_repeat = 1.1f;'
    )

for pattern, replacement in modifications:
    content = re.sub(pattern, replacement, content)

with open(cpp_file, 'w', encoding='utf-8') as f:
    f.write(content)

print("✅ Paramètres optimisés appliqués:")
print("   - n_ctx: 8192 tokens")
print("   - max_tokens: 256")
print("   - temperature: 0.7")
print("   - top_k: 50")
print("   - repetition_penalty: 1.1")

In [ ]:
# Cellule 7: Compilation de merlin_llm.dll
print("🔨 Compilation de merlin_llm.dll...\n")

os.chdir("/content/merlin_llm")

!rm -rf build
!mkdir build && cd build && \
cmake .. \
    -DCMAKE_TOOLCHAIN_FILE=/tmp/mingw-toolchain.cmake \
    -G Ninja \
    -DCMAKE_BUILD_TYPE=Release && \
ninja -v 2>&1 | tail -50

# Vérifier le résultat
dll_path = "/content/merlin_llm/addons/merlin_llm/bin/merlin_llm.windows.release.x86_64.dll"

if os.path.exists(dll_path):
    size_mb = os.path.getsize(dll_path) / (1024*1024)
    print(f"\n✅ COMPILATION RÉUSSIE!")
    print(f"   DLL: {dll_path}")
    print(f"   Taille: {size_mb:.2f} MB")
else:
    print("\n❌ Erreur: DLL non générée")
    !find /content/merlin_llm -name "*.dll" -o -name "*.so"
    raise Exception("DLL compilation failed")

In [ ]:
# Cellule 8: Téléchargement de la DLL compilée
from google.colab import files
import shutil

print("📦 Préparation du téléchargement...\n")

# Créer un ZIP avec la DLL et les fichiers modifiés
!mkdir -p /content/output
!cp /content/merlin_llm/addons/merlin_llm/bin/merlin_llm.windows.release.x86_64.dll /content/output/
!cp /content/merlin_llm/src/merlin_llm.cpp /content/output/merlin_llm.cpp.modified

# Créer le ZIP
os.chdir("/content")
!zip -r merlin_llm_compiled.zip output/

print("\n📥 Téléchargement de merlin_llm_compiled.zip...\n")
files.download("/content/merlin_llm_compiled.zip")

print("\n" + "="*60)
print("✅ COMPILATION TERMINÉE!")
print("="*60)
print("\nContenu du ZIP téléchargé:")
print("  - merlin_llm.windows.release.x86_64.dll")
print("  - merlin_llm.cpp.modified (pour référence)")
print("\nProchaines étapes sur votre PC:")
print("  1. Décompressez merlin_llm_compiled.zip")
print("  2. Copiez la DLL dans:")
print("     c:\\Users\\PGNK2128\\Godot-MCP\\addons\\merlin_llm\\bin\\")
print("  3. Fermez Godot si ouvert")
print("  4. Relancez Godot")
print("  5. Testez TestMerlinGBA.tscn")
print("\n🎁 Améliorations attendues:")
print("  - Réponses: 200-350 mots (vs 50-100)")
print("  - Créativité: 8/10 (vs 3/10)")
print("  - Contexte: 8192 tokens (stable!)")
print("  - Répétitions: quasi nulles")
print("\n🧙‍♂️ Votre Merlin est prêt!")